# Exercises — Trend-following strategies (EMA crossover)

[DataCamp exercise](https://campus.datacamp.com/courses/financial-trading-in-python/trading-strategies?ex=5) · see `Notes.md` in this folder for the summary.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project's data folder regardless of where this notebook runs from
DATA = next(p / "course materials" / "data"
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "course materials" / "data").is_dir())

def load(name):
    """Load an OHLCV CSV with a parsed DatetimeIndex."""
    return pd.read_csv(DATA / name, index_col="Date", parse_dates=True)

def price(name, col, year=None):
    """Single-asset price DataFrame (column = `col`) for use with bt."""
    df = load(name)
    if year:
        df = df[df.index.year == year]
    return df["Close"].rename(col).to_frame()


In [ ]:
import bt
import talib
import numpy as np

data = price("AMZN-stock-data.csv", "AMZN", year=2020)
EMA_short = talib.EMA(data["AMZN"], timeperiod=10)
EMA_long  = talib.EMA(data["AMZN"], timeperiod=40)

# +1 long when short>long, -1 short when short<long, 0 during warm-up
signal = pd.DataFrame(0.0, index=data.index, columns=["AMZN"])
signal["AMZN"] = np.where(EMA_short > EMA_long, 1.0,
                          np.where(EMA_short < EMA_long, -1.0, 0.0))
signal[EMA_long.isnull().values] = 0.0

bt_strategy = bt.Strategy("EMA_crossover", [
    bt.algos.WeighTarget(signal),
    bt.algos.Rebalance(),
])
bt_result = bt.run(bt.Backtest(bt_strategy, data))
bt_result.plot(title="EMA crossover (trend following)")
plt.show()